In [36]:
import tensorflow as tf
import numpy as np

print(tf.__version__)

2.22.0-rc0


In [37]:
tf.executing_eagerly()

True

In [38]:
# create tensor in TF
a = tf.constant(2.0)
b = tf.constant(3.0)
result = a + b
print(result)

tf.Tensor(5.0, shape=(), dtype=float32)


In [39]:
result.numpy()

np.float32(5.0)

#### compile(), fit(), evaluate(), and predict()
#### Sequential
#### Functional API & model subclassing (later for more flexibility)

In [40]:
texts = [
    "this movie was great",
    "i loved this film",
    "what a fantastic story",
    "the acting was wonderful",
    "this was an amazing movie",
    "i really enjoyed this",
    "the film was fun",
    "great story and great acting",
    "this movie was boring",
    "i hated this film",
    "what a terrible story",
    "the acting was awful",
    "this was a bad movie",
    "i really disliked this",
    "the film was dull",
    "boring story and bad acting"
]

labels = [
    1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0
]

In [41]:
len(texts)

16

In [42]:
len(labels)

16

In [43]:
texts[15], labels[15]

('boring story and bad acting', 0)

Convert Python Lists into TensorFlow Data

In [44]:
raw_dataset = tf.data.Dataset.from_tensor_slices((texts, labels))
raw_dataset

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>

In [45]:
for text, label in raw_dataset.take(3):
    print(text.numpy().decode("utf-8"), label.numpy())

this movie was great 1
i loved this film 1
what a fantastic story 1


In [46]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=8
)

In [47]:
vectorizer.adapt(texts)

In [48]:
vocab = vectorizer.get_vocabulary()

In [49]:
len(vocab)

29

In [50]:
vocab[:20]

['',
 '[UNK]',
 np.str_('was'),
 np.str_('this'),
 np.str_('the'),
 np.str_('story'),
 np.str_('movie'),
 np.str_('i'),
 np.str_('film'),
 np.str_('acting'),
 np.str_('great'),
 np.str_('a'),
 np.str_('what'),
 np.str_('really'),
 np.str_('boring'),
 np.str_('bad'),
 np.str_('and'),
 np.str_('wonderful'),
 np.str_('terrible'),
 np.str_('loved')]

In [51]:
example_text = tf.constant(["this movie was great"])
vectorized_example = vectorizer(example_text)
example_text.numpy()[0].decode("utf-8")

'this movie was great'

In [52]:
vectorized_example.numpy()

array([[ 3,  6,  2, 10,  0,  0,  0,  0]])

In [53]:
vectorized_example.shape

TensorShape([1, 8])

In [54]:
examples = tf.constant([
    "this movie was great",
    "sweet film",
    "what a fantastic story",
])

In [55]:
vectorized_examples = vectorizer(examples)

for original, vectorized in zip(examples, vectorized_examples):
    print(original.numpy().decode('utf-8'))
    print(vectorized.numpy())
    print()

this movie was great
[ 3  6  2 10  0  0  0  0]

sweet film
[1 8 0 0 0 0 0 0]

what a fantastic story
[12 11 22  5  0  0  0  0]



In [56]:
batch_size = 4

dataset = tf.data.Dataset.from_tensor_slices((texts, labels))
dataset = dataset.shuffle(buffer_size=len(texts), seed=42).batch(batch_size)

print(dataset)

<_BatchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.string, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [57]:
for batch_texts, batch_labels in dataset.take(1):
    print(batch_texts.shape)
    print(batch_labels.shape)
    print(batch_texts.numpy())
    print(batch_labels.numpy())

(4,)
(4,)
[b'i hated this film' b'this was a bad movie' b'i loved this film'
 b'boring story and bad acting']
[0 0 1 0]


In [58]:
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=1000, output_dim=16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (1, 8)                 │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [59]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [60]:
history = model.fit(
    dataset,
    epochs=20,
    verbose=1,
)

Epoch 1/20


C:\Users\OMID\Documents\PyEnv\PyEnv314\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5000 - loss: 0.6943  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 0.6925
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5000 - loss: 0.6915
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5000 - loss: 0.6908
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5625 - loss: 0.6894 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6250 - loss: 0.6883 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6875 - loss: 0.6876
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6875 - loss: 0.6865
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6875 - loss: 0.6855
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7500 - loss: 0.6843
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.6833
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.6823
Epoch 13/20
4/4 ━━━━

In [61]:
loss, accuracy = model.evaluate(dataset, verbose=0)

In [62]:
print(f"loss: {loss:.4f}, accuracy: {accuracy:.4f}")

loss: 0.6721, accuracy: 0.9375


In [64]:
new_texts = tf.constant([
    "this movie was great",
    "this was a terrible movie",
    "what a great movie",
    "what a terrible movie"
])

In [65]:
predictions = model.predict(new_texts)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


In [67]:
for text, score in zip(new_texts, predictions):
    text = text.numpy().decode("utf-8")
    score = float(score[0])
    label = "positive" if score > 0.5 else "negative"
    print(f"text: {text:30s}, score: {score:.4f}, {label}")

text: this movie was great          , score: 0.5186, positive
text: this was a terrible movie     , score: 0.4902, negative
text: what a great movie            , score: 0.5069, positive
text: what a terrible movie         , score: 0.4861, negative
